In [0]:
# ===========================================
# Notebook Name:
# 02_build_deck_silver
#
# Purpose:
# Parse raw Pokemon deck HTML and build
# normalized Silver Delta tables.
#
# Input:
# pokemon.bronze.deck_raw
#
# Output:
# pokemon.silver.decks
# pokemon.silver.deck_cards
#
# Processing:
# Bronze HTML
#   → Python parser
#   → Spark DataFrame
#   → Data quality validation
#   → Silver Delta
# ===========================================

In [0]:
%pip install beautifulsoup4

In [0]:
dbutils.library.restartPython()

In [0]:
import hashlib
from __future__ import annotations

from collections import defaultdict
from typing import Any
import re

from bs4 import BeautifulSoup

from pyspark.sql import functions as F
from pyspark.sql.types import (
    BooleanType,
    IntegerType,
    LongType,
    StringType,
    StructField,
    StructType,
    TimestampType,
)
from pyspark.sql.window import Window


DECK_RAW_TABLE = "pokemon.bronze.deck_raw"
DECK_TABLE = "pokemon.silver.decks"
DECK_CARD_TABLE = "pokemon.silver.deck_cards"

print("Input :", DECK_RAW_TABLE)
print("Output:", DECK_TABLE)
print("Output:", DECK_CARD_TABLE)

In [0]:
CATEGORY_NAMES = {
    "ポケモン": "pokemon",
    "グッズ": "goods",
    "ポケモンのどうぐ": "pokemon_tool",
    "サポート": "supporter",
    "スタジアム": "stadium",
    "エネルギー": "energy",
}


def normalize_text(value: str | None) -> str:
    if value is None:
        return ""

    return re.sub(
        r"\s+",
        " ",
        value.replace("\u3000", " "),
    ).strip()


def parse_quantity(value: str) -> int | None:
    normalized = normalize_text(value)

    if re.fullmatch(r"\d+", normalized):
        return int(normalized)

    match = re.fullmatch(
        r"(\d+)\s*枚",
        normalized,
    )

    if match:
        return int(match.group(1))

    return None

In [0]:
def parse_deck_html(
    html: str,
    deck_code: str,
) -> list[dict[str, Any]]:

    if not html:
        return []

    # Python標準のHTML parserを使用
    soup = BeautifulSoup(
        html,
        "html.parser",
    )

    cards: list[dict[str, Any]] = []
    current_category: str | None = None

    for row in soup.find_all("tr"):
        cells = [
            normalize_text(
                cell.get_text(
                    " ",
                    strip=True,
                )
            )
            for cell in row.find_all(
                ["th", "td"],
                recursive=False,
            )
        ]

        cells = [
            cell
            for cell in cells
            if cell
        ]

        if not cells:
            continue

        first_cell = cells[0]

        # カテゴリ見出しは完全一致だけで判定
        if first_cell in CATEGORY_NAMES:
            current_category = CATEGORY_NAMES[
                first_cell
            ]
            continue

        if current_category is None:
            continue

        # ヘッダー・小計・合計を除外
        if any(
            keyword in first_cell
            for keyword in [
                "カード名",
                "小計",
                "合計",
            ]
        ):
            continue

        quantity: int | None = None
        quantity_index: int | None = None

        for index, cell in enumerate(
            cells[1:],
            start=1,
        ):
            parsed_quantity = parse_quantity(cell)

            if parsed_quantity is not None:
                quantity = parsed_quantity
                quantity_index = index
                break

        if quantity is None or quantity_index is None:
            continue

        remaining_cells = cells[
            quantity_index + 1:
        ]

        expansion = (
            remaining_cells[0]
            if len(remaining_cells) >= 1
            else None
        )

        collection_number = (
            remaining_cells[1]
            if len(remaining_cells) >= 2
            else None
        )

        cards.append({
            "deck_code": deck_code,
            "category": current_category,
            "card_name": first_cell,
            "quantity": quantity,
            "expansion": expansion,
            "collection_number": collection_number,
        })

    return cards

In [0]:
def summarize_deck(
    cards: list[dict[str, Any]],
) -> dict[str, Any]:

    category_counts: dict[str, int] = defaultdict(int)

    for card in cards:
        category_counts[
            card["category"]
        ] += int(card["quantity"])

    total_cards = sum(
        int(card["quantity"])
        for card in cards
    )

    return {
        "total_cards": total_cards,
        "card_type_rows": len(cards),
        "unique_card_names": len({
            card["card_name"]
            for card in cards
        }),
        "category_counts": dict(
            category_counts
        ),
        "is_valid_60_cards": (
            total_cards == 60
        ),
    }

In [0]:
def build_deck_hash(
    cards: list[dict[str, Any]],
) -> tuple[str, str]:
    """
    Build a functional deck hash.

    Cards with the same normalized card name are aggregated
    regardless of expansion or collection number.

    Returns:
        deck_hash:
            SHA-256 hash of the canonical representation.

        canonical_string:
            Human-readable representation used for hashing.
    """
    quantity_by_card_name: dict[str, int] = defaultdict(int)

    for card in cards:
        normalized_card_name = normalize_text(
            card.get("card_name")
        )

        if not normalized_card_name:
            raise ValueError(
                "Card name is empty while generating deck hash"
            )

        quantity_by_card_name[
            normalized_card_name
        ] += int(card["quantity"])

    canonical_lines = [
        f"{card_name}|{quantity_by_card_name[card_name]}"
        for card_name in sorted(
            quantity_by_card_name.keys()
        )
    ]

    canonical_string = "\n".join(
        canonical_lines
    )

    deck_hash = hashlib.sha256(
        canonical_string.encode("utf-8")
    ).hexdigest()

    return deck_hash, canonical_string

In [0]:
latest_window = (
    Window
    .partitionBy("deck_code")
    .orderBy(
        F.col("scraped_at").desc(),
        F.col("response_hash").desc(),
    )
)

latest_deck_raw_df = (
    spark.table(DECK_RAW_TABLE)
    .withColumn(
        "_row_number",
        F.row_number().over(latest_window),
    )
    .filter(
        F.col("_row_number") == 1
    )
    .drop("_row_number")
)

display(
    latest_deck_raw_df.select(
        "deck_code",
        "response_hash",
        "scraped_at",
        F.length("payload").alias(
            "payload_length"
        ),
    )
    .orderBy("deck_code")
)

print(
    "Latest deck count:",
    latest_deck_raw_df.count(),
)

In [0]:
TARGET_DECK_CODE = "FdbkVk-ATjX03-FfVbFd"

target_row = (
    latest_deck_raw_df
    .filter(
        F.col("deck_code")
        == TARGET_DECK_CODE
    )
    .select(
        "deck_code",
        "payload",
    )
    .first()
)

if target_row is None:
    raise ValueError(
        f"Deck not found: {TARGET_DECK_CODE}"
    )

target_cards = parse_deck_html(
    html=target_row["payload"],
    deck_code=target_row["deck_code"],
)

target_summary = summarize_deck(
    target_cards
)

print(target_summary)

In [0]:
for card in target_cards:
    if card["category"] == "goods":
        print(
            card["card_name"],
            card["quantity"],
            card["expansion"],
            card["collection_number"],
        )

In [0]:
raw_deck_rows = (
    latest_deck_raw_df
    .select(
        "deck_code",
        "source_url",
        "response_hash",
        "scraped_at",
        "payload",
    )
    .collect()
)

all_deck_cards: list[dict[str, Any]] = []
deck_summaries: list[dict[str, Any]] = []
parse_errors: list[dict[str, str]] = []

for index, row in enumerate(
    raw_deck_rows,
    start=1,
):
    deck_code = row["deck_code"]

    print(
        f"[{index}/{len(raw_deck_rows)}] "
        f"Parsing {deck_code}"
    )

    try:
        cards = parse_deck_html(
            html=row["payload"],
            deck_code=deck_code,
        )

        summary = summarize_deck(cards)
        deck_hash, canonical_string = (
            build_deck_hash(cards)
        )

        for card in cards:
            all_deck_cards.append({
                **card,
                "deck_hash": deck_hash,
                "source_url": row["source_url"],
                "response_hash": row["response_hash"],
                "source_scraped_at": row["scraped_at"],
            })

        deck_summaries.append({
            "deck_hash": deck_hash,
            "deck_hash_version": "v1",
            "deck_code": deck_code,
            "source_url": row["source_url"],
            "response_hash": row["response_hash"],
            "source_scraped_at": row["scraped_at"],
            "total_cards": summary["total_cards"],
            "card_type_rows": summary["card_type_rows"],
            "unique_card_names": summary[
                "unique_card_names"
            ],
            "is_valid": summary[
                "is_valid_60_cards"
            ],
            "canonical_string": canonical_string,
        })

        print(
            "  total:",
            summary["total_cards"],
            "hash:",
            deck_hash[:12],
            "categories:",
            summary["category_counts"],
        )

    except Exception as error:
        parse_errors.append({
            "deck_code": deck_code,
            "error": str(error),
        })

        print("  ERROR:", error)

print("Parsed decks:", len(deck_summaries))
print("Card rows:", len(all_deck_cards))
print("Errors:", len(parse_errors))

In [0]:
if parse_errors:
    for error in parse_errors:
        print(error)

    raise ValueError(
        f"{len(parse_errors)} deck parsing errors occurred"
    )

print("No parsing errors.")

In [0]:
deck_card_schema = StructType([
    StructField(
        "deck_code",
        StringType(),
        False,
    ),
    StructField(
        "deck_hash",
        StringType(),
        False,
    ),
    StructField(
        "category",
        StringType(),
        False,
    ),
    StructField(
        "card_name",
        StringType(),
        False,
    ),
    StructField(
        "quantity",
        IntegerType(),
        False,
    ),
    StructField(
        "expansion",
        StringType(),
        True,
    ),
    StructField(
        "collection_number",
        StringType(),
        True,
    ),
    StructField(
        "source_url",
        StringType(),
        True,
    ),
    StructField(
        "response_hash",
        StringType(),
        True,
    ),
    StructField(
        "source_scraped_at",
        TimestampType(),
        True,
    ),
])


deck_schema = StructType([
    StructField(
        "deck_hash",
        StringType(),
        False,
    ),
    StructField(
        "deck_hash_version",
        StringType(),
        False,
    ),
    StructField(
        "deck_code",
        StringType(),
        False,
    ),
    StructField(
        "source_url",
        StringType(),
        True,
    ),
    StructField(
        "response_hash",
        StringType(),
        True,
    ),
    StructField(
        "source_scraped_at",
        TimestampType(),
        True,
    ),
    StructField(
        "total_cards",
        IntegerType(),
        False,
    ),
    StructField(
        "card_type_rows",
        LongType(),
        False,
    ),
    StructField(
        "unique_card_names",
        LongType(),
        False,
    ),
    StructField(
        "is_valid",
        BooleanType(),
        False,
    ),
    StructField(
        "canonical_string",
        StringType(),
        False,
    ),
])

In [0]:
if not all_deck_cards:
    raise ValueError(
        "No card rows were parsed"
    )

if not deck_summaries:
    raise ValueError(
        "No deck summaries were parsed"
    )

deck_cards_df = (
    spark.createDataFrame(
        all_deck_cards,
        schema=deck_card_schema,
    )
    .withColumn(
        "updated_at",
        F.current_timestamp(),
    )
)

decks_df = (
    spark.createDataFrame(
        deck_summaries,
        schema=deck_schema,
    )
    .withColumn(
        "updated_at",
        F.current_timestamp(),
    )
)

display(decks_df.orderBy("deck_code"))

In [0]:
deck_hash_validation_df = (
    decks_df
    .groupBy("deck_hash")
    .agg(
        F.countDistinct(
            "canonical_string"
        ).alias(
            "canonical_string_count"
        ),
        F.countDistinct(
            "deck_code"
        ).alias(
            "deck_code_count"
        ),
    )
)

display(
    deck_hash_validation_df.orderBy(
        F.col("deck_code_count").desc()
    )
)

In [0]:
deck_validation_df = (
    deck_cards_df
    .groupBy("deck_code")
    .agg(
        F.sum("quantity").alias(
            "total_cards"
        ),
        F.count("*").alias(
            "card_type_rows"
        ),
        F.countDistinct(
            "card_name"
        ).alias(
            "unique_card_names"
        ),
    )
    .withColumn(
        "is_valid",
        F.col("total_cards") == 60,
    )
)

display(
    deck_validation_df.orderBy(
        "deck_code"
    )
)

In [0]:
invalid_canonical_count = (
    canonical_quantity_df
    .filter(
        F.col("canonical_total_cards") != 60
    )
    .count()
)

if invalid_canonical_count > 0:
    raise ValueError(
        f"{invalid_canonical_count} canonical decks "
        "do not contain 60 cards"
    )

print(
    "Canonical representation validation passed."
)

In [0]:
invalid_decks_df = (
    deck_validation_df
    .filter(
        ~F.col("is_valid")
    )
)

invalid_count = invalid_decks_df.count()

if invalid_count > 0:
    display(invalid_decks_df)

    raise ValueError(
        f"{invalid_count} decks do not contain 60 cards"
    )

print(
    "Validation passed: "
    "all decks contain exactly 60 cards"
)


In [0]:
deck_category_validation_df = (
    deck_cards_df
    .groupBy(
        "deck_code",
        "category",
    )
    .agg(
        F.sum("quantity").alias(
            "card_count"
        )
    )
    .groupBy("deck_code")
    .pivot("category")
    .sum("card_count")
    .fillna(0)
    .orderBy("deck_code")
)

display(deck_category_validation_df)

In [0]:
display(
    deck_category_validation_df
    .filter(
        F.col("deck_code")
        == TARGET_DECK_CODE
    )
)

In [0]:
spark.sql(f"""
CREATE TABLE {DECK_TABLE}
(
    deck_hash STRING NOT NULL,
    deck_hash_version STRING NOT NULL,
    deck_code STRING NOT NULL,
    source_url STRING,
    response_hash STRING,
    source_scraped_at TIMESTAMP,
    total_cards INT,
    card_type_rows BIGINT,
    unique_card_names BIGINT,
    is_valid BOOLEAN,
    canonical_string STRING,
    updated_at TIMESTAMP
)
USING DELTA
COMMENT 'Normalized Pokemon deck compositions with functional deck hashes'
""")

In [0]:
spark.sql(f"""
CREATE TABLE {DECK_CARD_TABLE}
(
    deck_code STRING NOT NULL,
    deck_hash STRING NOT NULL,
    category STRING NOT NULL,
    card_name STRING NOT NULL,
    quantity INT NOT NULL,
    expansion STRING,
    collection_number STRING,
    source_url STRING,
    response_hash STRING,
    source_scraped_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
COMMENT 'Normalized card composition mapped to functional deck hashes'
""")

In [0]:
spark.sql(
    f"DROP TABLE IF EXISTS {DECK_CARD_TABLE}"
)

spark.sql(
    f"DROP TABLE IF EXISTS {DECK_TABLE}"
)

print("Old Silver deck tables dropped.")

In [0]:
(
    decks_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(DECK_TABLE)
)

(
    deck_cards_df.write
    .format("delta")
    .mode("append")
    .saveAsTable(DECK_CARD_TABLE)
)

print("Silver V3 deck tables created.")

In [0]:
display(
    spark.table(DECK_TABLE)
    .orderBy("deck_code")
)

In [0]:
display(
    spark.table(DECK_CARD_TABLE)
    .filter(
        F.col("deck_code")
        == TARGET_DECK_CODE
    )
    .orderBy(
        "category",
        "card_name",
    )
)

In [0]:
display(
    spark.table(DECK_CARD_TABLE)
    .filter(
        F.col("deck_code")
        == TARGET_DECK_CODE
    )
    .groupBy("category")
    .agg(
        F.sum("quantity").alias(
            "card_count"
        )
    )
    .orderBy("category")
)

In [0]:
target_deck_hash, target_canonical_string = (
    build_deck_hash(target_cards)
)

print("Deck hash:")
print(target_deck_hash)

print("\nCanonical string:")
print(target_canonical_string)

In [0]:
second_hash, _ = build_deck_hash(
    target_cards
)

assert target_deck_hash == second_hash

print("Hash stability validation passed.")

In [0]:
deck_hash_mapping_df = (
    spark.table(DECK_TABLE)
    .select(
        "deck_code",
        "deck_hash",
        "deck_hash_version",
    )
    .dropDuplicates(
        ["deck_code"]
    )
)

display(deck_hash_mapping_df)

In [0]:
spark.sql("""
ALTER TABLE pokemon.silver.tournament_results
ADD COLUMNS (
    deck_hash STRING,
    deck_hash_version STRING
)
""")

In [0]:
deck_hash_mapping_df.createOrReplaceTempView(
    "staged_deck_hash_mapping"
)

spark.sql("""
MERGE INTO pokemon.silver.tournament_results AS target
USING staged_deck_hash_mapping AS source
ON target.deck_code = source.deck_code

WHEN MATCHED THEN UPDATE SET
    target.deck_hash = source.deck_hash,
    target.deck_hash_version =
        source.deck_hash_version
""")

print(
    "tournament_results updated with deck_hash."
)

In [0]:
display(
    spark.table(DECK_TABLE)
    .select(
        "deck_hash",
        "deck_hash_version",
        "deck_code",
        "total_cards",
        "is_valid",
    )
    .orderBy("deck_code")
)

In [0]:
display(
    spark.table(DECK_CARD_TABLE)
    .select(
        "deck_hash",
        "deck_code",
        "category",
        "card_name",
        "quantity",
    )
    .orderBy(
        "deck_code",
        "category",
        "card_name",
    )
)

In [0]:
display(
    spark.table(
        "pokemon.silver.tournament_results"
    )
    .select(
        "tournament_id",
        "rank",
        "deck_code",
        "deck_hash",
        "deck_hash_version",
    )
    .orderBy(
        "tournament_id",
        "rank",
    )
)

In [0]:
missing_deck_hash_count = (
    spark.table(
        "pokemon.silver.tournament_results"
    )
    .filter(
        F.col("deck_code").isNotNull()
        & F.col("deck_hash").isNull()
    )
    .count()
)

if missing_deck_hash_count > 0:
    raise ValueError(
        f"{missing_deck_hash_count} tournament results "
        "are missing deck_hash"
    )

print(
    "Validation passed: "
    "all tournament decks have deck_hash"
)